# LeetCode 775 — Global and Local Inversions bằng Python

Link đề: https://leetcode.com/problems/global-and-local-inversions/ · Độ khó: **Medium** · Chủ đề: Array, Math

Skeleton LeetCode:

```python
class Solution:
    def isIdealPermutation(self, nums: List[int]) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với một hoán vị của `0..n-1`, so sánh **số nghịch thế toàn cục** (mọi cặp `i < j` mà `nums[i] > nums[j]`) với **số nghịch thế cục bộ** (chỉ các cặp **kề nhau** `j = i + 1`); trả `True` nếu hai số **bằng nhau**.
2. Nắm các công cụ Python: duyệt hai vòng lặp để đếm cặp, `max` chạy dần (running max), `abs`, `itertools.permutations` để vét cạn hoán vị nhỏ, và cấu trúc **cây chỉ số nhị phân (Fenwick tree / BIT)** để đếm nghịch thế nhanh.
3. Đi từ **đếm cả hai rồi so sánh (`O(n²)`)** → **đếm nghịch thế toàn cục bằng BIT (`O(n log n)`)** → **kiểm tra "không có nghịch thế xa" bằng max tiền tố (`O(n)`)** → **phát biểu gọn `|nums[i] - i| ≤ 1`**.
4. Hiểu mấu chốt: mỗi nghịch thế cục bộ **cũng là** nghịch thế toàn cục, nên "bằng nhau" ⇔ **không tồn tại nghịch thế không kề**; và với **hoán vị của `0..n-1`** điều đó tương đương `|nums[i] - i| ≤ 1` (có chứng minh), từ đó đếm được số hoán vị như vậy (số Fibonacci).
5. Tránh các lỗi kinh điển: so sánh `nums[i] == i` (quá chặt), tính max tiền tố gồm cả phần tử kề (`i-1`), nhầm "chỉ có một lần đổi chỗ", và viết `O(n²)` rồi bị TLE với `n = 10^5`.

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài `n` là một **hoán vị của mọi số nguyên trong `[0, n - 1]`**.

- **Nghịch thế toàn cục (global inversion):** số cặp **khác nhau** `(i, j)` với `0 <= i < j < n` và `nums[i] > nums[j]`.
- **Nghịch thế cục bộ (local inversion):** số chỉ số `i` với `0 <= i < n - 1` và `nums[i] > nums[i + 1]`.

Trả về `true` nếu **số nghịch thế toàn cục bằng số nghịch thế cục bộ**.

**Ví dụ 1:** `nums = [1,0,2]` ⇒ `true`. Có 1 nghịch thế toàn cục `(0,1)` và 1 nghịch thế cục bộ `i = 0`.

**Ví dụ 2:** `nums = [1,2,0]` ⇒ `false`. Có 2 nghịch thế toàn cục: `(0,2)` (1 > 0) và `(1,2)` (2 > 0); chỉ 1 nghịch thế cục bộ `i = 1`.

**Ràng buộc:**

- `n == nums.length`, `1 <= n <= 10^5`
- `0 <= nums[i] < n`, mọi giá trị **khác nhau**, `nums` là hoán vị của `0..n-1`.

Cell dưới nạp thư viện và đếm hai loại nghịch thế bằng cách trực tiếp cho hai ví dụ (chưa phải thuật toán tối ưu).

In [1]:
import random
import time
from itertools import permutations
from typing import List

examples = [([1, 0, 2], True), ([1, 2, 0], False)]
for nums, expected in examples:
    n = len(nums)
    glob = [(i, j) for i in range(n) for j in range(i + 1, n) if nums[i] > nums[j]]
    loc = [i for i in range(n - 1) if nums[i] > nums[i + 1]]
    print(nums, "| toàn cục:", glob, "| cục bộ:", loc, "->", len(glob) == len(loc), "(mong đợi", expected, ")")

[1, 0, 2] | toàn cục: [(0, 1)] | cục bộ: [0] -> True (mong đợi True )
[1, 2, 0] | toàn cục: [(0, 2), (1, 2)] | cục bộ: [1] -> False (mong đợi False )


## 3. Kiến thức nền

### 3.1. Nghịch thế (inversion) là gì?

Một **nghịch thế** là một cặp vị trí `i < j` mà giá trị ở trước **lớn hơn** giá trị ở sau — tức "ngược thứ tự tăng dần". Mảng đã sắp xếp tăng dần có `0` nghịch thế; mảng giảm dần hoàn toàn có `n(n-1)/2` nghịch thế. Số nghịch thế đo "mức độ lộn xộn" và bằng số lần đổi chỗ **hai phần tử kề nhau** tối thiểu để sắp xếp mảng.

**Cục bộ** là trường hợp đặc biệt khi hai vị trí **kề nhau** (`j = i + 1`). Vì mọi cặp kề nhau cũng là một cặp `(i, j)` với `i < j`, nên:

> **số nghịch thế cục bộ ≤ số nghịch thế toàn cục** (mỗi nghịch thế cục bộ là một nghịch thế toàn cục).

In [2]:
def count_global(nums):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n) if nums[i] > nums[j])


def count_local(nums):
    return sum(1 for i in range(len(nums) - 1) if nums[i] > nums[i + 1])


for arr in ([0, 1, 2, 3], [3, 2, 1, 0], [1, 0, 3, 2], [2, 0, 1]):
    print(arr, "toàn cục =", count_global(arr), "cục bộ =", count_local(arr))

[0, 1, 2, 3] toàn cục = 0 cục bộ = 0
[3, 2, 1, 0] toàn cục = 6 cục bộ = 3
[1, 0, 3, 2] toàn cục = 2 cục bộ = 2
[2, 0, 1] toàn cục = 2 cục bộ = 1


### 3.2. "Bằng nhau" nghĩa là gì?

Vì `local ⊆ global`, hai số bằng nhau khi và chỉ khi **mọi nghịch thế toàn cục đều là cục bộ**, tức là **không có** nghịch thế với `j ≥ i + 2` (hai vị trí cách nhau ít nhất 2). Gọi đó là **nghịch thế xa**. Bài toán trở thành: *mảng có nghịch thế xa không?*

### 3.3. Max chạy dần (running maximum)

Để biết "có phần tử nào ở các vị trí `0..i-2` lớn hơn `nums[i]` không", ta không cần xét từng phần tử: chỉ cần **giá trị lớn nhất** của đoạn `nums[0..i-2]`. Ta cập nhật `mx = max(mx, nums[i-2])` khi `i` tăng.

In [3]:
nums = [2, 0, 1, 4, 3]
mx = -1
for i in range(2, len(nums)):
    mx = max(mx, nums[i - 2])           # lớn nhất trong nums[0..i-2]
    print(f"i={i}: max(nums[0..{i - 2}]) = {mx}, nums[{i}] = {nums[i]}, nghịch thế xa? {mx > nums[i]}")

i=2: max(nums[0..0]) = 2, nums[2] = 1, nghịch thế xa? True
i=3: max(nums[0..1]) = 2, nums[3] = 4, nghịch thế xa? False
i=4: max(nums[0..2]) = 2, nums[4] = 3, nghịch thế xa? False


### 3.4. Hoán vị của `0..n-1` và `|nums[i] - i|`

Vì `nums` chứa **đúng một lần** mỗi số `0..n-1`, ta có thể so sánh giá trị `nums[i]` với **vị trí đúng của nó** `i` (trong mảng tăng dần `[0,1,...,n-1]`, `nums[i] = i`). Đại lượng `|nums[i] - i|` là "độ lệch khỏi vị trí đúng". Ở mục 6 ta chứng minh: không có nghịch thế xa ⇔ độ lệch mọi phần tử ≤ 1.

In [4]:
nums = [1, 0, 3, 2]
print([abs(x - i) for i, x in enumerate(nums)])        # [1, 1, 1, 1]
print([abs(x - i) for i, x in enumerate([2, 0, 1])])   # [2, 1, 1]  -> có phần tử lệch 2

[1, 1, 1, 1]
[2, 1, 1]


### 3.5. Cây chỉ số nhị phân (Fenwick tree / BIT) để đếm nghịch thế

Muốn đếm nghịch thế toàn cục trong `O(n log n)`: duyệt từ trái sang phải, với mỗi `x = nums[i]` đếm **có bao nhiêu số đã gặp lớn hơn `x`**. Cần cấu trúc hỗ trợ hai thao tác nhanh: *"thêm một giá trị"* và *"đếm các giá trị đã thêm ≤ `v`"*. BIT làm cả hai trong `O(log n)`:

- `add(p)`: tăng `tree[p]` và các ô cha `p += p & -p`.
- `query(p)`: tổng tiền tố `p`, đi `p -= p & -p`.

(`p & -p` lấy bit `1` thấp nhất của `p`.) Vị trí BIT đánh số từ 1 nên ta dùng `x + 1` cho giá trị `x ∈ [0, n-1]`.

In [5]:
class BIT:
    """Fenwick tree đếm tần suất: add(x) thêm giá trị x; count_le(x) = số giá trị đã thêm <= x."""

    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)

    def add(self, x):
        p = x + 1
        while p <= self.n:
            self.tree[p] += 1
            p += p & -p

    def count_le(self, x):
        p, s = x + 1, 0
        while p > 0:
            s += self.tree[p]
            p -= p & -p
        return s


bit = BIT(5)
for x in (3, 1, 4):
    bit.add(x)
print(bit.count_le(0), bit.count_le(1), bit.count_le(3), bit.count_le(4))     # 0 1 2 3

0 1 2 3


### 3.6. Vét cạn hoán vị nhỏ bằng `itertools.permutations`

`permutations(range(n))` sinh mọi hoán vị (`n!` cái). Với `n ≤ 8` (40 320 hoán vị) ta có thể **kiểm chứng mọi hoán vị** — công cụ rất mạnh để thử một nhận định về hoán vị.

In [6]:
print(len(list(permutations(range(4)))), list(permutations(range(3))))

24 [(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ví dụ 1:** `[1, 0, 2]`. Cặp `(0,1)`: `1 > 0` nghịch thế (kề nhau: cục bộ). Cặp `(0,2)`: `1 > 2`? Không. Cặp `(1,2)`: `0 > 2`? Không. ⇒ toàn cục = cục bộ = 1 ⇒ `true`.

**Ví dụ 2:** `[1, 2, 0]`. Cặp `(0,1)`: `1 > 2`? Không. `(1,2)`: `2 > 0` nghịch thế, kề nhau. `(0,2)`: `1 > 0` nghịch thế, **không kề** (cách nhau 2). ⇒ toàn cục 2, cục bộ 1 ⇒ `false`. Chính **nghịch thế xa** `(0,2)` phá vỡ đẳng thức.

**Hình dung:** hoán vị "lý tưởng" (ideal) là hoán vị mà phần tử nào "lạc chỗ" thì chỉ lạc **một bước** — kiểu `[1,0, 3,2, 4, 6,5]` (các cặp kề nhau đổi chỗ). Nếu có phần tử `x` ở trước một số nhỏ hơn mà cách ≥ 2 vị trí, ví dụ `2` đứng trước `0` hai vị trí (`[2,1,0]` hoặc `[2,0,1]`), thì không còn lý tưởng.

Các ví dụ tay:

| mảng | toàn cục | cục bộ | kết quả |
|---|---|---|---|
| `[0,1,2]` | 0 | 0 | true |
| `[1,0,3,2]` | 2 | 2 | true |
| `[2,0,1]` | 2 `(0,1),(0,2)` | 1 | false |
| `[0,2,1,3]` | 1 | 1 | true |
| `[3,2,1,0]` | 6 | 3 | false |

## 5. Cách 1 — Brute Force / baseline: đếm cả hai loại rồi so sánh

**Ý tưởng bám sát đề:** hai vòng lặp lồng nhau đếm nghịch thế toàn cục; một vòng lặp đếm nghịch thế cục bộ; trả `glob == loc`.

In [7]:
def ideal_bruteforce(nums):
    """Cách 1: đếm đúng theo định nghĩa. O(n^2)."""
    n = len(nums)
    glob = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] > nums[j]:
                glob += 1
    loc = 0
    for i in range(n - 1):
        if nums[i] > nums[i + 1]:
            loc += 1
    return glob == loc


for nums, expected in examples:
    print(nums, "->", ideal_bruteforce(nums), "(mong đợi", expected, ")")
    assert ideal_bruteforce(nums) == expected
print(ideal_bruteforce([0]), ideal_bruteforce([1, 0, 3, 2]), ideal_bruteforce([2, 0, 1]))     # True True False

[1, 0, 2] -> True (mong đợi True )
[1, 2, 0] -> False (mong đợi False )
True True False


**Độ phức tạp:** vòng đôi `n(n-1)/2` phép so sánh ⇒ **`O(n²)`** thời gian, `O(1)` bộ nhớ.

**Nút thắt:** `n = 10^5` ⇒ `5·10^9` phép so sánh, vượt xa giới hạn. Ta đang **đếm chính xác** hai số trong khi chỉ cần biết chúng **có bằng nhau không**. Hai hướng cải thiện: (1) đếm nghịch thế toàn cục nhanh hơn bằng cấu trúc dữ liệu (Cách 2); (2) tận dụng `local ⊆ global` để chỉ cần kiểm tra **không có nghịch thế xa** (Cách 3, 4).

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — `local ⊆ global`.** Mỗi nghịch thế cục bộ là một nghịch thế toàn cục ⇒ `local ≤ global`; hai số **bằng nhau** ⇔ **không có nghịch thế xa** (cặp `i + 2 ≤ j` với `nums[i] > nums[j]`).

**Bước 2 — kiểm tra "không có nghịch thế xa" bằng max tiền tố.** Với mỗi `j ≥ 2`: có nghịch thế xa kết thúc ở `j` ⇔ `max(nums[0..j-2]) > nums[j]`. Giữ `mx` chạy dần ⇒ một lần quét `O(n)`: **nếu tồn tại `j` có `mx > nums[j]` ⇒ `False`**.

**Bước 3 — dùng thêm tính chất hoán vị: `|nums[i] - i| ≤ 1`.** Chứng minh tương đương (với hoán vị của `0..n-1`):

*(⇒)* Giả sử `nums[i] ≥ i + 2` với một `i`. Có `nums[i] ≥ i + 2` giá trị **nhỏ hơn** `nums[i]` (đó là `0, 1, ..., nums[i]-1`). Chỉ có `i` vị trí đứng trước `i`, nên **ít nhất 2** trong các giá trị nhỏ đó nằm **sau** vị trí `i`. Mỗi giá trị như thế tạo nghịch thế với `nums[i]`; mà chỉ **một** vị trí (`i + 1`) là kề, nên có ít nhất một nghịch thế xa ⇒ không lý tưởng. Tương tự nếu `nums[i] ≤ i - 2`: có `n - 1 - nums[i] ≥ n - i + 1` giá trị **lớn hơn** `nums[i]`, nhưng chỉ có `n - 1 - i` vị trí sau `i`, nên ít nhất 2 giá trị lớn hơn nằm **trước** `i`; mỗi giá trị đó ở vị trí `k < i` tạo nghịch thế `(k, i)`, và chỉ `k = i - 1` là kề nên có ít nhất một nghịch thế xa. Vậy lý tưởng ⇒ `|nums[i] - i| ≤ 1` với mọi `i`.

*(⇐)* Giả sử `|nums[k] - k| ≤ 1` với mọi `k` và có `i + 2 ≤ j`. Khi đó `nums[i] ≤ i + 1 ≤ j - 1 ≤ nums[j]`; hai giá trị khác nhau nên `nums[i] < nums[j]` — không có nghịch thế xa ✓.

⇒ **Chỉ cần kiểm tra `abs(nums[i] - i) <= 1` cho mọi `i`** — `O(n)`, `O(1)` bộ nhớ, rất gọn.

(Ta kiểm chứng toàn bộ nhận định bằng vét cạn **mọi hoán vị với `n ≤ 8`** ở mục 12.)

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

- Cách 1: đếm cả hai rồi so sánh (mục 5).
- Cách 2: đếm nghịch thế toàn cục bằng BIT (`O(n log n)`), so với số cục bộ.
- Cách 3: max tiền tố — kiểm tra không có nghịch thế xa (**chính thức**).
- Cách 4: kiểm tra `|nums[i] - i| ≤ 1`.

---

### Cách 2 — Đếm nghịch thế toàn cục bằng BIT

Giải thích từng đoạn:

1. Duyệt `i` từ trái sang phải; BIT lưu các giá trị đã gặp.
2. Số nghịch thế kết thúc ở `i` = (số phần tử đã gặp) − (số phần tử đã gặp `≤ nums[i]`) = `i - bit.count_le(nums[i])`.
3. Cộng vào `glob`, rồi `bit.add(nums[i])`.
4. Đếm `loc` (nghịch thế cục bộ) bằng một vòng quét.
5. Trả `glob == loc`.

In [8]:
def ideal_bit(nums):
    """Cách 2: đếm nghịch thế toàn cục bằng BIT, O(n log n)."""
    n = len(nums)
    bit = BIT(n)
    glob = 0
    for i, x in enumerate(nums):
        glob += i - bit.count_le(x)           # số phần tử đã gặp (i) mà lớn hơn x
        bit.add(x)
    loc = sum(1 for i in range(n - 1) if nums[i] > nums[i + 1])
    return glob == loc


for nums, expected in examples:
    print(nums, "->", ideal_bit(nums))
    assert ideal_bit(nums) == expected
print(ideal_bit([2, 0, 1]), ideal_bit([1, 0, 3, 2]))        # False True

[1, 0, 2] -> True
[1, 2, 0] -> False
False True


**Code nộp LeetCode cho Cách 2:**

```python
class Solution:
    def isIdealPermutation(self, nums: List[int]) -> bool:
        n = len(nums)
        tree = [0] * (n + 1)

        def add(p):
            while p <= n:
                tree[p] += 1
                p += p & -p

        def query(p):
            s = 0
            while p > 0:
                s += tree[p]
                p -= p & -p
            return s

        glob = 0
        for i, x in enumerate(nums):
            glob += i - query(x + 1)
            add(x + 1)
        loc = sum(nums[i] > nums[i + 1] for i in range(n - 1))
        return glob == loc
```

**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ. Tổng quát hơn (đếm nghịch thế cho **mọi** mảng, không cần hoán vị đặc biệt), nhưng chậm hơn các cách sau.

---

### Cách 3 — Max tiền tố: không có nghịch thế xa (chính thức)

Giải thích từng đoạn:

1. `mx = 0` (hoặc `-1`): lớn nhất trong `nums[0..i-2]`.
2. Với `i` từ `2` đến `n - 1`: cập nhật `mx = max(mx, nums[i - 2])`.
3. Nếu `mx > nums[i]` ⇒ có nghịch thế xa `(vị trí đạt mx, i)` ⇒ trả `False`.
4. Quét hết mà không có ⇒ `True`.

In [9]:
def ideal_prefix_max(nums):
    """Cách 3: nghịch thế xa tồn tại <=> max(nums[0..i-2]) > nums[i] với một i >= 2."""
    mx = 0
    for i in range(2, len(nums)):
        mx = max(mx, nums[i - 2])
        if mx > nums[i]:
            return False
    return True


for nums, expected in examples:
    print(nums, "->", ideal_prefix_max(nums))
    assert ideal_prefix_max(nums) == expected
print(ideal_prefix_max([0]), ideal_prefix_max([1, 0]), ideal_prefix_max([2, 0, 1]))        # True True False

[1, 0, 2] -> True
[1, 2, 0] -> False
True True False


**Code nộp LeetCode cho Cách 3:** (đó là code ở mục 9.)

**Độ phức tạp:** **`O(n)`** thời gian, **`O(1)`** bộ nhớ.

---

### Cách 4 — Kiểm tra `|nums[i] - i| ≤ 1`

Giải thích từng đoạn (nhờ chứng minh ở mục 6):

1. Với mỗi vị trí `i`, tính độ lệch `abs(nums[i] - i)`.
2. Nếu có độ lệch `> 1` ⇒ `False`.
3. Ngược lại `True`.

Có thể viết một dòng: `all(abs(x - i) <= 1 for i, x in enumerate(nums))`.

In [10]:
def ideal_displacement(nums):
    """Cách 4: hoán vị của 0..n-1 là lý tưởng <=> |nums[i] - i| <= 1 với mọi i."""
    return all(abs(x - i) <= 1 for i, x in enumerate(nums))


for nums, expected in examples:
    print(nums, "->", ideal_displacement(nums))
    assert ideal_displacement(nums) == expected
print(ideal_displacement([0]), ideal_displacement([1, 0, 3, 2]), ideal_displacement([2, 0, 1]))    # True True False

[1, 0, 2] -> True
[1, 2, 0] -> False
True True False


**Code nộp LeetCode cho Cách 4:**

```python
class Solution:
    def isIdealPermutation(self, nums: List[int]) -> bool:
        return all(abs(x - i) <= 1 for i, x in enumerate(nums))
```

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ. **Lưu ý:** cách này **chỉ đúng vì** `nums` là hoán vị của `0..n-1` (đề đảm bảo). Cách 3 đúng cho cả mảng bất kỳ các giá trị khác nhau (chỉ cần so sánh), nên "an toàn" hơn nếu đề đổi.

## 8. TRACE từng bước

### 8.1. Trace Cách 3 với `nums = [1, 2, 0]` (đáp án `false`)

| i | `nums[i-2]` | `mx` sau cập nhật | `nums[i]` | `mx > nums[i]`? | hành động |
|---|---|---|---|---|---|
| 2 | `nums[0] = 1` | `max(0, 1) = 1` | 0 | `1 > 0` đúng | **trả `False`** |

(Nghịch thế xa chính là `(0, 2)`: `1 > 0`.)

### 8.2. Trace Cách 3 với `nums = [1, 0, 3, 2, 5, 4]` (đáp án `true`)

| i | `nums[i-2]` | `mx` | `nums[i]` | `mx > nums[i]`? |
|---|---|---|---|---|
| 2 | `nums[0] = 1` | 1 | 3 | không |
| 3 | `nums[1] = 0` | 1 | 2 | không |
| 4 | `nums[2] = 3` | 3 | 5 | không |
| 5 | `nums[3] = 2` | 3 | 4 | không |

Không bao giờ `mx > nums[i]` ⇒ `True`. Hàm `trace_prefix_max` in các dòng này.

In [11]:
def trace_prefix_max(nums):
    print("nums =", nums)
    mx = 0
    for i in range(2, len(nums)):
        mx = max(mx, nums[i - 2])
        far = mx > nums[i]
        print(f"i={i}: nums[i-2]={nums[i - 2]} -> mx={mx}, nums[i]={nums[i]}, mx > nums[i]? {far}")
        if far:
            print("=> có nghịch thế xa, trả False")
            return False
    print("=> không có nghịch thế xa, trả True")
    return True


assert trace_prefix_max([1, 2, 0]) is False
print("----")
assert trace_prefix_max([1, 0, 3, 2, 5, 4]) is True

nums = [1, 2, 0]
i=2: nums[i-2]=1 -> mx=1, nums[i]=0, mx > nums[i]? True
=> có nghịch thế xa, trả False
----
nums = [1, 0, 3, 2, 5, 4]
i=2: nums[i-2]=1 -> mx=1, nums[i]=3, mx > nums[i]? False
i=3: nums[i-2]=0 -> mx=1, nums[i]=2, mx > nums[i]? False
i=4: nums[i-2]=3 -> mx=3, nums[i]=5, mx > nums[i]? False
i=5: nums[i-2]=2 -> mx=3, nums[i]=4, mx > nums[i]? False
=> không có nghịch thế xa, trả True


### 8.3. Trace Cách 2 (BIT) với `nums = [1, 2, 0]`

| i | x = `nums[i]` | đã gặp | `count_le(x)` | nghịch thế mới `i - count_le` | `glob` |
|---|---|---|---|---|---|
| 0 | 1 | {} | 0 | 0 - 0 = 0 | 0 |
| 1 | 2 | {1} | 1 (số 1 ≤ 2) | 1 - 1 = 0 | 0 |
| 2 | 0 | {1, 2} | 0 | 2 - 0 = 2 | 2 |

`glob = 2`, `loc = 1` (`2 > 0`) ⇒ `False` ✓.

In [12]:
def trace_bit(nums):
    n = len(nums)
    bit = BIT(n)
    glob = 0
    for i, x in enumerate(nums):
        le = bit.count_le(x)
        glob += i - le
        print(f"i={i}: x={x}, đã gặp {i} số, trong đó <= x có {le} -> nghịch thế mới {i - le}, glob={glob}")
        bit.add(x)
    loc = sum(1 for i in range(n - 1) if nums[i] > nums[i + 1])
    print(f"glob={glob}, loc={loc} -> {glob == loc}")
    return glob == loc


assert trace_bit([1, 2, 0]) is False

i=0: x=1, đã gặp 0 số, trong đó <= x có 0 -> nghịch thế mới 0, glob=0
i=1: x=2, đã gặp 1 số, trong đó <= x có 1 -> nghịch thế mới 0, glob=0
i=2: x=0, đã gặp 2 số, trong đó <= x có 0 -> nghịch thế mới 2, glob=2
glob=2, loc=1 -> False


## 9. Code nộp LeetCode

Chọn Cách 3 (max tiền tố — kiểm tra trực tiếp "không có nghịch thế xa", đúng cho mọi mảng các số khác nhau): đúng `class Solution`, phương thức `isIdealPermutation(self, nums: List[int]) -> bool`.

In [13]:
class Solution:
    def isIdealPermutation(self, nums: List[int]) -> bool:
        mx = 0
        for i in range(2, len(nums)):
            mx = max(mx, nums[i - 2])
            if mx > nums[i]:
                return False
        return True


sol = Solution()
print(sol.isIdealPermutation([1, 0, 2]))     # True
print(sol.isIdealPermutation([1, 2, 0]))     # False

True
False


## 10. Vì sao đúng

1. **`local ⊆ global`:** mỗi cặp `(i, i+1)` với `nums[i] > nums[i+1]` là một cặp `(i, j)` hợp lệ của nghịch thế toàn cục. Do đó `local ≤ global`, và `global - local` = số **nghịch thế xa** (cặp `i + 2 ≤ j`, `nums[i] > nums[j]`).
2. **Đẳng thức ⇔ không có nghịch thế xa:** vì `global - local = (số nghịch thế xa) ≥ 0`, đẳng thức xảy ra ⇔ số nghịch thế xa bằng 0.
3. **Cách 3 đúng:** có nghịch thế xa kết thúc ở `j` (`j ≥ 2`) ⇔ tồn tại `i ≤ j - 2` với `nums[i] > nums[j]` ⇔ `max(nums[0..j-2]) > nums[j]`. Duyệt `j` tăng dần, `mx` luôn là max của `nums[0..j-2]` (cập nhật thêm `nums[j-2]`); gặp `mx > nums[j]` thì có nghịch thế xa ⇒ `False`; quét hết ⇒ không có ⇒ `True`.
4. **Cách 4 đúng:** nhờ phép chứng minh ở mục 6 (đếm các giá trị nhỏ hơn/lớn hơn so với số vị trí còn lại ở một phía, và chiều ngược lại dựa trên tính khác nhau của các giá trị). Kiểm chứng thêm bằng vét cạn mọi hoán vị `n ≤ 8`.
5. **Cách 2 đúng:** `i - count_le(x)` là số phần tử đã gặp lớn hơn `x` (các giá trị khác nhau nên "lớn hơn" = "không ≤"), cộng dồn được `global` chính xác; so với `local`.
6. **Biên:** `n = 1, 2` không có `i ≥ 2` nên vòng lặp Cách 3 không chạy ⇒ `True`; điều này đúng vì với `n ≤ 2` mọi nghịch thế (nếu có) đều kề nhau.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đếm hai loại | `O(n²)` | `O(1)` | chỉ để học / oracle |
| 2. BIT đếm nghịch thế | `O(n log n)` | `O(n)` | đúng cho mọi mảng khác nhau từng đôi |
| 3. Max tiền tố | `O(n)` | `O(1)` | **nộp bài** |
| 4. `|nums[i] - i| ≤ 1` | `O(n)` | `O(1)` | gọn, dựa vào hoán vị của `0..n-1` |

Với `n = 10^5`: Cách 1 ≈ `5·10^9` phép so sánh (không thể); Cách 2 ≈ `10^5 · 17 · 2` bước BIT; Cách 3, 4 chỉ `10^5` bước.

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề** (cả 4 cách + `Solution`).
2. **Edge case:** `n = 1`, `n = 2` (cả hai hoán vị), mảng đã sắp xếp, mảng giảm dần, một phần tử lạc xa, nhiều cặp đổi chỗ kề nhau.
3. **Vét cạn mọi hoán vị của `0..n-1` với `n ≤ 8`**: các cách đều khớp oracle Cách 1.
4. **Hoán vị lý tưởng sinh theo cấu trúc** (các cặp kề nhau đổi chỗ ngẫu nhiên) ⇒ `True`; rồi **phá hỏng** bằng đổi chỗ hai phần tử cách xa ⇒ `False`.
5. **Ngẫu nhiên có seed đúng ràng buộc:** `n` tới `10^5`.
6. **Hiệu năng.**

In [14]:
IMPLS = [ideal_bruteforce, ideal_bit, ideal_prefix_max, ideal_displacement, lambda a: Solution().isIdealPermutation(a)]
NAMES = ["bruteforce", "bit", "prefix_max", "displacement", "Solution"]

cases = [
    ([1, 0, 2], True),
    ([1, 2, 0], False),
    ([0], True),
    ([0, 1], True),
    ([1, 0], True),
    ([0, 1, 2, 3, 4], True),            # đã sắp xếp
    ([3, 2, 1, 0], False),              # giảm dần
    ([2, 0, 1], False),
    ([0, 2, 1, 3], True),
    ([1, 0, 3, 2], True),
    ([1, 0, 3, 2, 5, 4], True),
    ([0, 3, 1, 2], False),              # số 3 lạc xa
    ([4, 0, 1, 2, 3], False),
    ([1, 2, 3, 4, 0], False),
]
for nums, expected in cases:
    for f, name in zip(IMPLS, NAMES):
        got = f(list(nums))
        assert got == expected, (name, nums, got, expected)
print(len(cases), "ca cố định: cả 5 cách đều đúng")

# Vét cạn: MỌI hoán vị của 0..n-1 với n <= 8
total = ideal_count = 0
for n in range(1, 9):
    for perm in permutations(range(n)):
        arr = list(perm)
        expected = ideal_bruteforce(arr)
        for f, name in zip(IMPLS[1:], NAMES[1:]):
            assert f(arr) == expected, (name, arr)
        total += 1
        ideal_count += expected
print("vét cạn", total, "hoán vị (n <= 8): mọi cách khớp oracle; trong đó", ideal_count, "hoán vị lý tưởng")

14 ca cố định: cả 5 cách đều đúng


vét cạn 46233 hoán vị (n <= 8): mọi cách khớp oracle; trong đó 87 hoán vị lý tưởng


In [15]:
# Hoán vị lý tưởng sinh theo cấu trúc: lát các khối [x] hoặc [x+1, x]; rồi phá hỏng
def make_ideal(rng, n):
    arr, i = [], 0
    while i < n:
        if i + 1 < n and rng.random() < 0.5:
            arr += [i + 1, i]               # đổi chỗ hai vị trí kề nhau
            i += 2
        else:
            arr.append(i)
            i += 1
    return arr


rng = random.Random(775)
for _ in range(2000):
    n = rng.randint(1, 30)
    arr = make_ideal(rng, n)
    assert sorted(arr) == list(range(n))
    for f, name in zip(IMPLS, NAMES):
        assert f(list(arr)) is True, (name, arr)
    if n >= 3:                                # phá hỏng: đổi chỗ hai vị trí cách nhau >= 2 (giá trị khác nhau nên luôn đổi thứ tự)
        i = rng.randint(0, n - 3)
        j = rng.randint(i + 2, n - 1)
        bad = arr[:]
        bad[i], bad[j] = bad[j], bad[i]
        expected = ideal_bruteforce(bad)
        for f, name in zip(IMPLS[1:], NAMES[1:]):
            assert f(list(bad)) == expected, (name, bad)
print("2000 hoán vị lý tưởng sinh theo cấu trúc: True; bản bị phá: khớp oracle")

# Ngẫu nhiên đúng ràng buộc: n tới 10^5. Cách 2, 3, 4, Solution so nhau (Cách 1 chỉ cho n nhỏ ở trên)
for _ in range(30):
    n = rng.choice([1, 2, 3, 10, 1000, 10 ** 5])
    arr = make_ideal(rng, n)
    if rng.random() < 0.5 and n >= 3:
        i = rng.randint(0, n - 3)
        j = rng.randint(i + 2, n - 1)
        arr[i], arr[j] = arr[j], arr[i]
    res = {name: f(list(arr)) for f, name in zip(IMPLS[1:], NAMES[1:])}
    assert len(set(res.values())) == 1, (n, res)
print("30 input lớn (n tới 10^5): Cách 2 = Cách 3 = Cách 4 = Solution")

arr = list(range(10 ** 5))
random.Random(1).shuffle(arr)
res = {name: f(list(arr)) for f, name in zip(IMPLS[1:], NAMES[1:])}
assert set(res.values()) == {False}
print("hoán vị ngẫu nhiên hoàn toàn n = 10^5 -> False cho mọi cách")

2000 hoán vị lý tưởng sinh theo cấu trúc: True; bản bị phá: khớp oracle


30 input lớn (n tới 10^5): Cách 2 = Cách 3 = Cách 4 = Solution


hoán vị ngẫu nhiên hoàn toàn n = 10^5 -> False cho mọi cách


In [16]:
# Hiệu năng
n = 10 ** 5
ideal_arr = make_ideal(random.Random(2), n)
shuffled = list(range(n)); random.Random(3).shuffle(shuffled)
for label, arr in (("lý tưởng n=10^5", ideal_arr), ("ngẫu nhiên n=10^5", shuffled)):
    row = []
    for f, name in zip(IMPLS[1:], NAMES[1:]):
        t0 = time.perf_counter()
        r = f(list(arr))
        row.append(f"{name}: {r} {time.perf_counter() - t0:.4f}s")
    print(f"{label:<20} " + " | ".join(row))

for size in (500, 1000, 2000):
    arr = make_ideal(random.Random(size), size)
    t0 = time.perf_counter()
    ideal_bruteforce(arr)
    print(f"Cách 1, n={size}: {time.perf_counter() - t0:.4f}s  (gấp đôi n ~ gấp 4 thời gian -> n = 10^5 không thể)")

lý tưởng n=10^5      bit: True 0.1804s | prefix_max: True 0.0082s | displacement: True 0.0066s | Solution: True 0.0079s
ngẫu nhiên n=10^5    bit: False 0.1912s | prefix_max: False 0.0006s | displacement: False 0.0005s | Solution: False 0.0004s
Cách 1, n=500: 0.0035s  (gấp đôi n ~ gấp 4 thời gian -> n = 10^5 không thể)


Cách 1, n=1000: 0.0146s  (gấp đôi n ~ gấp 4 thời gian -> n = 10^5 không thể)
Cách 1, n=2000: 0.0603s  (gấp đôi n ~ gấp 4 thời gian -> n = 10^5 không thể)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đòi `nums[i] == i` (quá chặt)

"Lý tưởng" nghĩa là chỉ lạc **một** bước, không phải **không lạc**. Với `[1, 0, 2]` đề trả `true` nhưng `nums[0] != 0`.

In [17]:
def wrong_strict(nums):
    return all(x == i for i, x in enumerate(nums))      # SAI: chỉ chấp nhận mảng đã sắp xếp


print(wrong_strict([1, 0, 2]), "(đúng: True)")
print(wrong_strict([0, 1, 2]), "(đúng: True)")
print(wrong_strict([1, 0, 3, 2]), "(đúng: True)")

False (đúng: True)
True (đúng: True)
False (đúng: True)


**Sửa:** cho phép lệch `1`: `abs(nums[i] - i) <= 1`.

### Lỗi 2: max tiền tố bao gồm cả phần tử kề (`i - 1`)

Nghịch thế **kề nhau được phép**. Nếu `mx = max(nums[0..i-1])` rồi so với `nums[i]`, ta coi cả nghịch thế cục bộ là "xa" ⇒ trả `False` oan.

In [18]:
def wrong_prefix_includes_adjacent(nums):
    mx = 0
    for i in range(1, len(nums)):
        mx = max(mx, nums[i - 1])            # SAI: gồm cả nums[i-1] (kề nhau)
        if mx > nums[i]:
            return False
    return True


print(wrong_prefix_includes_adjacent([1, 0, 2]), "(đúng: True)")
print(wrong_prefix_includes_adjacent([1, 0, 3, 2]), "(đúng: True)")
print(wrong_prefix_includes_adjacent([0, 1, 2]), "(đúng: True)")

False (đúng: True)
False (đúng: True)
True (đúng: True)


**Sửa:** chỉ gộp `nums[0..i-2]`: cập nhật `mx = max(mx, nums[i - 2])` và so với `nums[i]`, bắt đầu từ `i = 2`.

### Lỗi 3: nhầm thành "chỉ có đúng một lần đổi chỗ kề nhau"

Nhìn ví dụ `[1,0,2]` dễ kết luận "ideal ⇔ ít nhất/nhất định **một** cặp bị đổi". Thực ra có thể có **nhiều cặp rời nhau** (`[1,0,3,2]` vẫn `true`).

In [19]:
def wrong_single_swap(nums):
    local = sum(1 for i in range(len(nums) - 1) if nums[i] > nums[i + 1])
    return local <= 1                          # SAI: tưởng chỉ được tối đa một nghịch thế cục bộ


print(wrong_single_swap([1, 0, 3, 2]), "(đúng: True — hai cặp đổi chỗ rời nhau)")
print(wrong_single_swap([1, 0, 3, 2, 5, 4]), "(đúng: True)")
print(wrong_single_swap([2, 0, 1]), "(đúng: False, nhưng sai lý do)")
print(wrong_single_swap([1, 2, 0]), "(đúng: False)")

False (đúng: True — hai cặp đổi chỗ rời nhau)
False (đúng: True)
True (đúng: False, nhưng sai lý do)
True (đúng: False)


**Vì sao sai:** `local ≤ 1` bỏ sót các hoán vị lý tưởng nhiều cặp, và còn trả `True` cho hoán vị có nghịch thế xa nhưng chỉ một nghịch thế cục bộ (`[1, 2, 0]` có `local = 1` nhưng `global = 2`). **Sửa:** so với `global`, hoặc kiểm tra không có nghịch thế xa.

### Lỗi 4: viết `O(n²)` (đếm cả hai) rồi bị Time Limit Exceeded

Với `n = 10^5`, hai vòng lặp lồng nhau không thể chạy. Hãy nhận ra dấu hiệu "ràng buộc `10^5` ⇒ cần `O(n)` hoặc `O(n log n)`" và tìm cách dùng cấu trúc đặc biệt của đề (hoán vị, `local ⊆ global`).

In [20]:
for size in (1000, 2000, 4000):
    arr = list(range(size))
    t0 = time.perf_counter()
    ideal_bruteforce(arr)
    t1 = time.perf_counter() - t0
    t0 = time.perf_counter()
    ideal_prefix_max(arr)
    t2 = time.perf_counter() - t0
    print(f"n={size}: O(n^2) {t1:.4f}s  vs  O(n) {t2:.6f}s")

n=1000: O(n^2) 0.0145s  vs  O(n) 0.000131s
n=2000: O(n^2) 0.0583s  vs  O(n) 0.000135s


n=4000: O(n^2) 0.2617s  vs  O(n) 0.000292s


**Checklist chống lỗi:**

- [ ] Hiểu `local ⊆ global` ⇒ chỉ cần chặn **nghịch thế xa**.
- [ ] Max tiền tố chỉ đến `i - 2` (không gồm `i - 1`).
- [ ] Không đòi `nums[i] == i`.
- [ ] Không giới hạn "một cặp đổi chỗ".
- [ ] Chọn thuật toán `O(n)` cho `n = 10^5`.

## 14. Mẹo & nhận dạng pattern

- **"So sánh hai đại lượng mà một đại lượng là tập con của đại lượng kia"** ⇒ biến thành "có phần tử thừa không?" (ở đây: có nghịch thế xa không).
- **"Có phần tử nào ở trước, cách xa ≥ 2, lớn hơn phần tử hiện tại?"** ⇒ **max tiền tố lệch hai ô** (`mx` đến `i-2`). Biến thể "nhỏ hơn" dùng min hậu tố.
- **Hoán vị của `0..n-1`** thường cho phép **so sánh `nums[i]` với `i`** (độ lệch/dịch chuyển) — "mỗi phần tử cách vị trí đúng nhiều nhất 1".
- **Đếm nghịch thế tổng quát:** BIT hoặc merge sort (`O(n log n)`); số nghịch thế = số lần đổi chỗ kề nhau tối thiểu để sắp xếp.
- **Ràng buộc `10^5`:** không dùng `O(n²)`.
- **Kiểm chứng nhận định bằng vét cạn hoán vị nhỏ** (`itertools.permutations`) trước khi tin.
- Bài họ hàng: 493 (Reverse Pairs), 315 (Count of Smaller Numbers After Self), 1585, 1649, 768/769 (Max Chunks To Make Sorted).

## 15. Biến thể

1. **Đếm số hoán vị lý tưởng của `0..n-1`:** là số Fibonacci — xem bài tập 5. (Phần vét cạn ở mục 12 cho `1, 2, 3, 5, 8, ...`.)
2. **Số nghịch thế xa** `= global - local`: đếm bằng BIT trừ đi số cục bộ.
3. **Mảng bất kỳ (không phải hoán vị của `0..n-1`)** với các số khác nhau: Cách 3 vẫn đúng; Cách 4 không.
4. **Có số trùng nhau:** định nghĩa nghịch thế vẫn là `>` chặt, Cách 3 vẫn đúng với `mx > nums[i]` chặt; Cách 4 không còn đúng.
5. **Số lần đổi chỗ kề nhau tối thiểu để sắp xếp** = số nghịch thế toàn cục (đếm bằng BIT/merge sort).
6. **Hoán vị chỉ cách vị trí đúng tối đa `k`** (`|nums[i] - i| ≤ k`): tổng quát hoá Cách 4.

Minh hoạ biến thể 1 (đếm bằng vét cạn và bằng công thức Fibonacci) và 2:

In [21]:
def fib_count(n):
    a, b = 1, 1                       # a(0) = 1, a(1) = 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b if n >= 1 else 1


print([sum(1 for p in permutations(range(n)) if ideal_prefix_max(list(p))) for n in range(1, 9)])
print([fib_count(n) for n in range(1, 9)])                       # 1 2 3 5 8 13 21 34
assert all(fib_count(n) == sum(1 for p in permutations(range(n)) if ideal_prefix_max(list(p))) for n in range(1, 9))


def far_inversions(nums):
    """Biến thể 2: số nghịch thế xa = toàn cục - cục bộ (toàn cục đếm bằng BIT)."""
    n = len(nums)
    bit = BIT(n)
    glob = 0
    for i, x in enumerate(nums):
        glob += i - bit.count_le(x)
        bit.add(x)
    return glob - sum(1 for i in range(n - 1) if nums[i] > nums[i + 1])


print(far_inversions([1, 2, 0]), far_inversions([1, 0, 3, 2]), far_inversions([3, 2, 1, 0]))     # 1 0 3

[1, 2, 3, 5, 8, 13, 21, 34]
[1, 2, 3, 5, 8, 13, 21, 34]
1 0 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đếm hai loại | đúng định nghĩa, hai vòng lặp | `O(n²)` | `O(1)` | oracle, `n` nhỏ |
| 2. BIT | đếm nghịch thế toàn cục nhanh rồi so với cục bộ | `O(n log n)` | `O(n)` | khi cần **số** nghịch thế, mảng bất kỳ |
| 3. Max tiền tố | không có nghịch thế xa ⇔ `max(nums[0..i-2]) ≤ nums[i]` | `O(n)` | `O(1)` | **nộp bài**, đúng cho mọi mảng khác nhau từng đôi |
| 4. `|nums[i]-i| ≤ 1` | tính chất đặc biệt của hoán vị `0..n-1` | `O(n)` | `O(1)` | ngắn nhất; chỉ đúng với hoán vị `0..n-1` |

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có cell TODO để bạn tự làm (cell **chưa** được gọi, chỉ gán `None` hoặc `def` rỗng). Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Dự đoán giá trị `count_global` và `count_local` (ở mục 3.1) cho `[2, 1, 0]`, rồi kết quả của `Solution().isIdealPermutation([2, 1, 0])` và `Solution().isIdealPermutation([0, 2, 1, 3])`. Ghi vào list `bai1_du_doan = [toan_cuc, cuc_bo, kq1, kq2]`.

In [22]:
# TODO Bài 1: bai1_du_doan = [toan_cuc, cuc_bo, kq1, kq2]
bai1_du_doan = None

### Bài 2 (tính tay)

`nums = [2, 0, 1, 3]`. Liệt kê các cặp nghịch thế toàn cục và cục bộ; chỉ ra nghịch thế xa (nếu có). Kết quả ghi vào `bai2 = (so_toan_cuc, so_cuc_bo, ket_qua_bool)`.

In [23]:
# TODO Bài 2: bai2 = (so_toan_cuc, so_cuc_bo, ket_qua)
bai2 = None

### Bài 3 (điền TODO)

Hoàn thành hàm `is_ideal(nums)` bằng max tiền tố (cập nhật `mx` với `nums[i - 2]`).

In [24]:
def is_ideal(nums):
    # TODO: mx = 0; for i in range(2, len(nums)): mx = max(mx, nums[i - 2]); nếu mx > nums[i] trả False; cuối cùng trả True
    pass

### Bài 4 (tự viết lại)

Không nhìn mục 9, tự viết `is_ideal_permutation(nums)` theo Cách 4 (độ lệch `|nums[i] - i|`).

In [25]:
# TODO Bài 4: tự viết lại
def is_ideal_permutation(nums):
    pass

### Bài 5 (biến thể / mini challenge)

Viết `count_ideal(n)` trả số hoán vị lý tưởng của `0..n-1` (kết quả là số Fibonacci). Gợi ý đệ quy: phần tử đầu `0` đứng yên (còn `n-1` phần tử) hoặc `0` và `1` đổi chỗ (còn `n-2` phần tử). Kiểm chứng với vét cạn cho `n ≤ 8`.

In [26]:
# TODO Bài 5
def count_ideal(n):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `[2,1,0]` giảm dần hoàn toàn, mọi cặp là nghịch thế: `n(n-1)/2`.
- Mức 2: cục bộ chỉ đếm cặp kề nhau: `(0,1)` và `(1,2)`.
- Mức 3: `[3, 2, False, True]`: với `[0,2,1,3]` chỉ có một nghịch thế `(1,2)` vừa toàn cục vừa cục bộ.

**Bài 2**
- Mức 1: các cặp `(i,j)` có `nums[i] > nums[j]`: so `2` với `0`, `1`.
- Mức 2: `(0,1)`: 2>0 (kề); `(0,2)`: 2>1 (cách 2); các cặp còn lại không nghịch thế.
- Mức 3: `(2, 1, False)`; nghịch thế xa là `(0,2)`.

**Bài 3**
- Mức 1: bắt đầu `i = 2`.
- Mức 2: `mx = max(mx, nums[i - 2])`.
- Mức 3: `if mx > nums[i]: return False`; sau vòng lặp `return True`.

**Bài 4**
- Mức 1: một dòng với `all`.
- Mức 2: `enumerate(nums)` cho `(i, x)`.
- Mức 3: `return all(abs(x - i) <= 1 for i, x in enumerate(nums))`.

**Bài 5**
- Mức 1: `f(0) = 1`, `f(1) = 1`.
- Mức 2: `f(n) = f(n-1) + f(n-2)`.
- Mức 3: dùng hai biến cuộn `a, b = b, a + b`; kiểm chứng với `permutations`.

## 19. Đáp án tham khảo

Chạy kiểm tra từng đáp án bằng `assert`.

In [27]:
# Bài 1
dap_an_1 = [count_global([2, 1, 0]), count_local([2, 1, 0]),
            Solution().isIdealPermutation([2, 1, 0]), Solution().isIdealPermutation([0, 2, 1, 3])]
print(dap_an_1)
assert dap_an_1 == [3, 2, False, True]

# Bài 2
arr = [2, 0, 1, 3]
bai2_dap_an = (count_global(arr), count_local(arr), Solution().isIdealPermutation(arr))
print(bai2_dap_an)
assert bai2_dap_an == (2, 1, False)

# Bài 3
def is_ideal(nums):
    mx = 0
    for i in range(2, len(nums)):
        mx = max(mx, nums[i - 2])
        if mx > nums[i]:
            return False
    return True


for nums, expected in cases:
    assert is_ideal(nums) == expected
print("Bài 3: OK")

[3, 2, False, True]
(2, 1, False)
Bài 3: OK


In [28]:
# Bài 4
def is_ideal_permutation(nums):
    return all(abs(x - i) <= 1 for i, x in enumerate(nums))


for nums, expected in cases:
    assert is_ideal_permutation(nums) == expected
for n in range(1, 8):
    for p in permutations(range(n)):
        assert is_ideal_permutation(list(p)) == ideal_bruteforce(list(p))
print("Bài 4: OK")

# Bài 5
def count_ideal(n):
    a, b = 1, 1                          # f(0) = 1, f(1) = 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b


assert [count_ideal(n) for n in range(1, 9)] == [1, 2, 3, 5, 8, 13, 21, 34]
for n in range(1, 9):
    assert count_ideal(n) == sum(1 for p in permutations(range(n)) if ideal_bruteforce(list(p)))
print("Bài 5: OK")

Bài 4: OK
Bài 5: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "không có nghịch thế xa" (max tiền tố lệch hai ô):**

```python
class Solution:
    def isIdealPermutation(self, nums: List[int]) -> bool:
        mx = 0                                   # lớn nhất trong nums[0..i-2]
        for i in range(2, len(nums)):
            mx = max(mx, nums[i - 2])            # thêm phần tử cách 2 ô
            if mx > nums[i]:                     # có nghịch thế xa (không kề nhau)
                return False
        return True
```

Phiên bản ngắn dựa vào tính chất hoán vị `0..n-1`:

```python
return all(abs(x - i) <= 1 for i, x in enumerate(nums))
```

**Checklist trước khi submit:**

- [ ] Hiểu `local ⊆ global` ⇒ "bằng nhau" ⇔ không có nghịch thế xa?
- [ ] Max tiền tố chỉ gồm `nums[0..i-2]` (không gồm `i-1`), và vòng lặp bắt đầu từ `i = 2`?
- [ ] Không đòi `nums[i] == i`, không giới hạn "một cặp đổi chỗ"?
- [ ] Độ phức tạp `O(n)` cho `n = 10^5`?
- [ ] Đã test: `n = 1, 2`, sắp xếp sẵn, giảm dần, nhiều cặp đổi chỗ kề nhau, một phần tử lạc xa?
- [ ] Đã vét cạn mọi hoán vị nhỏ (`n ≤ 8`) so với oracle đếm hai loại?